# QUELL - Step 03: Preprocessing + adil bolme (TEK HUCRE, NIHAI)

Uc dataseti de bir kerede, dogru stratejilerle isler:
- **Edge-IIoTset**: temporal (frame.time).
- **CICIoT2023**: awk ile hizli ornek -> birebir dedup -> stratified (oturum kimligi yok).
- **N-BaIoT**: gerekiyorsa Kaggle'dan tam surum indir -> session-aware (kaynak fileya gore).

Cikti: `data/processed/`, `splits/`, `results/split_report.json`. Tek hucreyi Shift+Enter, sonunda OZET'i paylas.

In [ ]:
import os, json, glob, subprocess, sys, shutil
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd

ROOT = Path.home()/"quell-edge-llm-ids"
RAW  = ROOT/"data"/"raw"
PROC = ROOT/"data"/"processed"; PROC.mkdir(parents=True, exist_ok=True)
SPL  = ROOT/"splits"; SPL.mkdir(parents=True, exist_ok=True)
(ROOT/"results").mkdir(parents=True, exist_ok=True)
SEED=42; RATIOS=(0.6,0.2,0.2); MAX_ROWS=1_500_000
try: import pyarrow; PARQUET=True
except Exception:
    subprocess.run([sys.executable,"-m","pip","install","-q","pyarrow"])
    try: import pyarrow; PARQUET=True
    except Exception: PARQUET=False
print("parquet:",PARQUET,"| RAW:",RAW, flush=True)

LABEL_CANDS=["label","attack","attack_type","attack_label","type","class","category","marker"]
TIME_CANDS =["ts","timestamp","time","flow_start","start_time","stime","frame.time","date","datetime"]
LEAKY_HINT =["ip","mac","addr","src_","dst_","source","destination"]

def guess(cols,cands):
    low={c.lower():c for c in cols}
    for k in cands:
        if k in low: return low[k]
    for c in cols:
        if any(k in c.lower() for k in cands): return c
    return None

def make_split(df,label,time_col=None,group_col=None,perclass=False):
    n=len(df); a=int(n*RATIOS[0]); b=int(n*(RATIOS[0]+RATIOS[1])); rng=np.random.default_rng(SEED)
    if perclass and time_col and time_col in df.columns:
        # per-class temporal: each class ordered by time within itself (all classes appear in train)
        tv=df[time_col].values; y=df[label].astype(str).values; tr=[];va=[];te=[]
        for cls in pd.unique(y):
            idx=np.where(y==cls)[0]; order=idx[np.argsort(tv[idx],kind="mergesort")]
            ca=int(len(order)*RATIOS[0]); cb=int(len(order)*(RATIOS[0]+RATIOS[1]))
            tr+=order[:ca].tolist(); va+=order[ca:cb].tolist(); te+=order[cb:].tolist()
        return np.array(tr),np.array(va),np.array(te), f"per-class temporal (within-class {time_col})"
    if time_col and time_col in df.columns:
        order=np.argsort(df[time_col].values,kind="mergesort")
        return order[:a],order[a:b],order[b:], f"temporal (sirala: {time_col})"
    if group_col and group_col in df.columns:
        g=df[group_col].astype(str).values; uniq=np.array(sorted(set(g))); rng.shuffle(uniq)
        ga=int(len(uniq)*RATIOS[0]); gb=int(len(uniq)*(RATIOS[0]+RATIOS[1]))
        s1,s2,s3=set(uniq[:ga]),set(uniq[ga:gb]),set(uniq[gb:]); idx=np.arange(n)
        return idx[np.isin(g,list(s1))],idx[np.isin(g,list(s2))],idx[np.isin(g,list(s3))], f"grouped/session-aware (group: {group_col}, {len(uniq)} file)"
    buckets=defaultdict(list)
    for i,y in enumerate(df[label].astype(str).values): buckets[y].append(i)
    tr,va,te=[],[],[]
    for y,ids in buckets.items():
        ids=np.array(ids); rng.shuffle(ids); na=int(len(ids)*RATIOS[0]); nb=int(len(ids)*(RATIOS[0]+RATIOS[1]))
        tr+=list(ids[:na]); va+=list(ids[na:nb]); te+=list(ids[nb:])
    return np.array(tr),np.array(va),np.array(te), "stratified"

REPORT={}
def process(name,df,label,time_col=None,group_col=None,dedup=False,perclass=False,note=""):
    df=df.replace([np.inf,-np.inf],np.nan)
    ded=0
    if dedup:
        before=len(df); feat=[c for c in df.columns if c!=label]
        df=df.drop_duplicates(subset=feat,keep="first").reset_index(drop=True); ded=before-len(df)
    leaky=[c for c in df.columns if any(h in c.lower() for h in LEAKY_HINT)]
    tr,va,te,strat=make_split(df,label,time_col,group_col,perclass=perclass)
    if dedup: strat=f"{strat} + birebir dedup ({ded} tekrar cikti)"
    if note: strat=f"{strat}; {note}"
    if PARQUET: df.to_parquet(PROC/f"{name}.parquet"); fmt="parquet"
    else: df.to_csv(PROC/f"{name}.csv.gz",index=False,compression="gzip"); fmt="csv.gz"
    np.savez_compressed(SPL/f"{name}_split.npz",train=tr,val=va,test=te)
    meta={"n_rows":int(len(df)),"n_cols":int(df.shape[1]),"label_col":label,"strategy":strat,
          "time_col":time_col,"group_col":group_col,"dedup_removed":int(ded),
          "leaky_candidates":leaky[:15],"stored_as":fmt,"n_classes":int(df[label].nunique()),
          "split_sizes":{"train":int(len(tr)),"val":int(len(va)),"test":int(len(te))},
          "class_dist_overall":df[label].astype(str).value_counts().head(40).to_dict()}
    REPORT[name]=meta
    print(f"\n===== {name} =====", flush=True)
    print("rows:",len(df),"| column:",df.shape[1],"| label:",label,"| class:",meta["n_classes"])
    print("STRATEJI:",strat,"| bolme:",meta["split_sizes"])
    if leaky: print("leakage candidates:",leaky[:10])
    print("dagilim:",json.dumps(meta["class_dist_overall"],ensure_ascii=False))

# ================= Edge-IIoTset (temporal) =================
try:
    f=(glob.glob(str(RAW/"edge_iiotset"/"**"/"ML-EdgeIIoT-dataset.csv"),recursive=True) or
       glob.glob(str(RAW/"edge_iiotset"/"**"/"DNN-EdgeIIoT-dataset.csv"),recursive=True))[0]
    df=pd.read_csv(f,low_memory=False)
    process("edge_iiotset",df,guess(df.columns,["attack_type"]) or guess(df.columns,LABEL_CANDS),
            time_col=guess(df.columns,TIME_CANDS), perclass=True,
            note="per-class because a global temporal split separates the classes")
except Exception as e: print("EDGE HATA:",e, flush=True)

# ================= CICIoT2023 (awk subsample -> dedup -> stratified) =================
try:
    f=max(glob.glob(str(RAW/"ciciot2023"/"**/*.csv"),recursive=True),key=os.path.getsize)
    total=int(subprocess.check_output(f'wc -l < "{f}"',shell=True))-1
    k=max(1,round(total/MAX_ROWS)); tmp=str(PROC/"_ciciot_sample.csv")
    print(f"\nCICIoT ~{total:,} rows; every {k}. rows (awk, ~1-3 dk)...", flush=True)
    subprocess.run(f"awk 'NR==1 || NR % {k} == 0' \"{f}\" > \"{tmp}\"", shell=True, check=True)
    df=pd.read_csv(tmp,low_memory=False); os.remove(tmp)
    print("  CICIoT sample row:",len(df), flush=True)
    process("ciciot2023",df,guess(df.columns,LABEL_CANDS),dedup=True,
            note="CICIoT contains no session id (part-XXXXX random partitions)")
except Exception as e: print("CICIOT HATA:",e, flush=True)

# ================= N-BaIoT (Kaggle full version -> session-aware) =================
try:
    nbd=RAW/"nbaiot"
    have=(glob.glob(str(nbd/"**"/"*mirai*.csv"),recursive=True) or
          glob.glob(str(nbd/"**"/"*gafgyt*.csv"),recursive=True))
    if not have:
        print("\nNo N-BaIoT attack files -> downloading the full version from Kaggle (mkashifn/nbaiot-dataset)...", flush=True)
        shutil.rmtree(nbd,ignore_errors=True); nbd.mkdir(parents=True,exist_ok=True)
        subprocess.run(f'kaggle datasets download -d mkashifn/nbaiot-dataset -p "{nbd}" --unzip',shell=True)
    def nbc(p):
        p=p.lower()
        if "benign" in p: return "benign"
        bot="mirai" if "mirai" in p else ("gafgyt" if "gafgyt" in p else "bashlite")
        for k in ("combo","junk","udpplain","scan","tcp","udp","ack","syn"):
            if k in p: return f"{bot}_{k}"
        return "unknown"
    files=[x for x in glob.glob(str(nbd/"**"/"*.csv"),recursive=True)
           if os.path.getsize(x)>1000 and any(t in os.path.basename(x).lower() for t in ("benign","gafgyt","mirai","bashlite"))]
    print("N-BaIoT valid files:",len(files), flush=True)
    fr=[pd.read_csv(fp,low_memory=False).assign(__label__=nbc(fp),__srcfile__=os.path.basename(fp)) for fp in files]
    df=pd.concat(fr,ignore_index=True); del fr
    if len(df)>MAX_ROWS:
        df=df.groupby("__label__",group_keys=False).sample(frac=MAX_ROWS/len(df),random_state=SEED).reset_index(drop=True)
    process("nbaiot",df,"__label__",group_col="__srcfile__")
except Exception as e: print("NBAIOT HATA:",e, flush=True)

# ================= OZET =================
json.dump(REPORT,open(ROOT/"results"/"split_report.json","w"),indent=2,default=str,ensure_ascii=False)
print("\n=================  OZET  =================", flush=True)
for k,v in REPORT.items():
    print(f"{k}: {v['n_rows']} rows, {v['n_classes']} class | {v['strategy']} | {v['split_sizes']}")
print("saved -> results/split_report.json")
